# Week 6 Day 5 – Optimized Fine-Tuning for Price Prediction

This notebook applies **optimization techniques** to the Day 5 fine-tuning pipeline to reduce **average absolute error** on the test set.

**Run from:** `week6` directory (so `pricer` imports work).

**Baseline (Day 5):** ~82–96 error with 100 train / 50 val, simple prompt, 1 epoch, batch_size=1.

**Optimizations:**
1. **Richer prompt** – Include category and Amazon US context
2. **More training data** – 200 train / 50 val (more signal without heavy overfitting)
3. **System prompt** – Clear instruction for consistent output format
4. **Hyperparameters** – `batch_size=4`, `learning_rate_multiplier=0.5`
5. **Stratified sampling** – Balance price bands and categories in train/val

In [ ]:
import os
import sys

# Add week6 to path so pricer can be imported (works from repo root, week6, or community-contributions)
for _p in [".", os.path.join(os.getcwd(), ".."), os.path.join(os.getcwd(), "week6")]:
    _p = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_p, "pricer")):
        sys.path.insert(0, _p)
        break

import re
import json
import random
from collections import defaultdict
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)
openai = OpenAI()

In [ ]:
LITE_MODE = False
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

## Optimization 1: Stratified sampling

Ensure train/val include a balanced mix of price bands and categories.

In [ ]:
def price_band(p):
    if p < 50: return "low"
    if p < 150: return "mid"
    return "high"

def stratified_sample(items, n_train=200, n_val=50, seed=42):
    random.seed(seed)
    by_strata = defaultdict(list)
    for item in items:
        key = (item.category, price_band(item.price))
        by_strata[key].append(item)
    train_out, val_out = [], []
    ratio = n_train / (n_train + n_val)
    for stratum, pool in by_strata.items():
        random.shuffle(pool)
        n = len(pool)
        split = max(0, min(n - 1, int(n * ratio)))
        train_out.extend(pool[:split])
        val_out.extend(pool[split:])
    random.shuffle(train_out)
    random.shuffle(val_out)
    fine_tune_train = train_out[:n_train]
    fine_tune_val = val_out[:n_val] if len(val_out) >= n_val else val_out
    return fine_tune_train, fine_tune_val

fine_tune_train, fine_tune_val = stratified_sample(train + val, n_train=200, n_val=50)
print(f"Optimized split: {len(fine_tune_train)} train, {len(fine_tune_val)} val")

## Optimization 2 & 3: Richer prompt + system message

Include category and Amazon US context; add a system prompt for consistent output format.

In [ ]:
SYSTEM_PROMPT = "You are a product pricing expert. Estimate the price in USD. Respond with only the price in format $X.XX, no other text."

def messages_for_optimized(item):
    user_msg = (
        f"Category: {item.category}\n\n"
        f"Product: {item.summary}\n\n"
        f"Estimate the price in USD. Use Amazon US 2023 as reference. Respond with the price only, no explanation."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

def test_messages_for_optimized(item):
    user_msg = (
        f"Category: {item.category}\n\n"
        f"Product: {item.summary}\n\n"
        f"Estimate the price in USD. Use Amazon US 2023 as reference. Respond with the price only, no explanation."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]

## Create JSONL and upload

In [ ]:
def make_jsonl_optimized(items):
    lines = []
    for item in items:
        messages = messages_for_optimized(item)
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)

os.makedirs("jsonl", exist_ok=True)

with open("jsonl/optimized_train.jsonl", "w") as f:
    f.write(make_jsonl_optimized(fine_tune_train))
with open("jsonl/optimized_validation.jsonl", "w") as f:
    f.write(make_jsonl_optimized(fine_tune_val))

print(f"Wrote jsonl/optimized_train.jsonl ({len(fine_tune_train)} examples)")
print(f"Wrote jsonl/optimized_validation.jsonl ({len(fine_tune_val)} examples)")

In [ ]:
with open("jsonl/optimized_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open("jsonl/optimized_validation.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")
print(f"Uploaded: {train_file.id}, {val_file.id}")

## Optimization 4: Fine-tune with tuned hyperparameters

`batch_size=4`, `learning_rate_multiplier=0.5`, `n_epochs=1`

In [ ]:
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={
        "n_epochs": 1,
        "batch_size": 4,
        "learning_rate_multiplier": 0.5,
    },
    suffix="pricer-optimized",
)
print(f"Job: {job.id}")

## Wait for job completion

In [ ]:
import time

job_id = job.id
while True:
    status = openai.fine_tuning.jobs.retrieve(job_id)
    print(status.status)
    if status.status == "succeeded":
        fine_tuned_model_name = status.fine_tuned_model
        print(f"Model: {fine_tuned_model_name}")
        break
    elif status.status == "failed":
        print(status)
        break
    time.sleep(30)

## Inference and evaluation

In [ ]:
def optimized_pricer(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for_optimized(item),
        max_tokens=10,
        seed=42,
    )
    return response.choices[0].message.content

In [ ]:
evaluate(optimized_pricer, test, size=200)

## Baseline comparison (optional)

Run this to compare with the Day 5 baseline style (100 train, simple prompt). If you have a baseline fine-tuned model ID, set it below.

In [ ]:
def messages_baseline(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}, {"role": "assistant", "content": f"${item.price:.2f}"}]

def test_messages_baseline(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]

# Uncomment and set your baseline model to compare:
BASELINE_MODEL = "ft:gpt-4.1-nano-..."
def baseline_pricer(item):
    r = openai.chat.completions.create(model=BASELINE_MODEL, messages=test_messages_baseline(item), max_tokens=7)
    return r.choices[0].message.content
evaluate(baseline_pricer, test, size=200)